In [2]:
# Imports needed
import pandas as pd
import json
import sklearn
import nltk
import seaborn as sns

In [3]:
# Load only the first 10,000 rows from the dataset
review_data = pd.read_json('yelp_academic_dataset_review.json', lines=True, nrows=10000)

# Show columns to identify location fields
print("Columns in dataset:", review_data.columns.tolist())

# Display the first 10,000 rows
print(f"Loaded {len(review_data)} rows")
print("\nFirst 10,000 rows of the dataset:")
#review_data

# Load only the first # rows from the dataset
# Get all business data to make sure when merging, there are some guaranteed business ID matches from review data read in
business_data = pd.read_json('yelp_academic_dataset_business.json', lines=True)#, nrows=25)

# Show columns to identify location fields
print("Columns in dataset:", business_data.columns.tolist())

# Display the first # rows
print(f"Loaded {len(business_data)} rows")
print("\nFirst # rows of the dataset:")
#business_data

# Only keep businesses that appear in reviews
business_data = business_data[business_data['business_id'].isin(review_data['business_id'])]

Columns in dataset: ['review_id', 'user_id', 'business_id', 'stars', 'useful', 'funny', 'cool', 'text', 'date']
Loaded 10000 rows

First 10,000 rows of the dataset:
Columns in dataset: ['business_id', 'name', 'address', 'city', 'state', 'postal_code', 'latitude', 'longitude', 'stars', 'review_count', 'is_open', 'attributes', 'categories', 'hours']
Loaded 150346 rows

First # rows of the dataset:


## Cuisine Types

In [4]:
# Only include restaurants (remove doctors, shipping centers, etc.)
business_data = business_data[business_data['categories'].str.contains('Restaurants', na=False)]

def get_cuisine(categories):
    categories = str(categories)

    if 'Chinese' in categories:
        return 'Chinese'
    elif 'Italian' in categories:
        return 'Italian'
    elif 'Mexican' in categories:
        return 'Mexican'
    elif 'French' in categories:
        return 'French'
    elif 'Japanese' in categories or 'Sushi' in categories:
        return 'Japanese'
    elif 'Korean' in categories:
        return 'Korean'
    elif 'Mediterranean' in categories:
        return 'Mediterranean'
    elif 'Vietnamese' in categories:
        return 'Vietnamese'
    elif 'American' in categories or 'Burgers' in categories or 'Fast Food' in categories:
        return 'American'
    else:
        return None

In [5]:
print("Review rows:", len(review_data))
print("Business rows:", len(business_data))
print(business_data['categories'].head(30))

# Merge two datasets (review, business)

business_data['cuisine_label'] = business_data['categories'].apply(get_cuisine)

merged_data = review_data.merge(business_data[['business_id', 'cuisine_label']], on='business_id')

print("After merge:", len(merged_data))
print("Null labels after merge:", merged_data['cuisine_label'].isna().sum())

# Drop rows with no label
merged_data = merged_data.dropna(subset=['cuisine_label'])
print("After filtering:", len(merged_data))

Review rows: 10000
Business rows: 2210
3      Restaurants, Food, Bubble Tea, Coffee & Tea, B...
14           Food, Delis, Italian, Bakeries, Restaurants
15                     Sushi Bars, Restaurants, Japanese
19                                   Korean, Restaurants
20     Coffee & Tea, Food, Cafes, Bars, Wine Bars, Re...
23                                  Restaurants, Italian
28     Cocktail Bars, Bars, Italian, Nightlife, Resta...
31                       Pizza, Restaurants, Salad, Soup
33                                    Pizza, Restaurants
41     Restaurants, Specialty Food, Steakhouses, Food...
45                                  Restaurants, Chinese
47     Coffee & Tea, Restaurants, Wine Bars, Bars, Ni...
53     Coffee & Tea, Cafes, Pets, Restaurants, Pet Ad...
59                                    Restaurants, Pizza
60                   Restaurants, Soup, Seafood, Burgers
61     Sports Bars, American (New), American (Traditi...
64     Seafood, Restaurants, Bars, Nightlife, Coc

## Logistic Regression with Unigram

#### Split into training and test sets

In [6]:
from sklearn.model_selection import train_test_split

print("Dataset size:", len(merged_data))
print(merged_data['cuisine_label'].value_counts())

x_input = merged_data['text']
y_output = merged_data['cuisine_label']

test_size = int(0.2 * len(merged_data))
x_train, x_test, y_train, y_test = train_test_split(x_input, y_output, test_size=test_size, random_state=9)
print(len(x_train), len(x_test))
print(len(y_train), len(y_test))

Dataset size: 5012
cuisine_label
American         2498
Italian           646
Mexican           625
Japanese          419
Chinese           303
Mediterranean     208
French            167
Vietnamese        102
Korean             44
Name: count, dtype: int64
4010 1002
4010 1002


## Extract Unigram Features

In [7]:
from sklearn.feature_extraction.text import CountVectorizer

unigram_vectorizer = CountVectorizer()
unigram_vectorizer.fit(x_train)
train_features = unigram_vectorizer.transform(x_train)
test_features = unigram_vectorizer.transform(x_test)

print(train_features.shape) # prints (number of rows in the matrix, number of columns)
print(test_features.shape)  # prints (number of rows in the matrix, number of columns)

(4010, 14362)
(1002, 14362)


## Train and Evaluate

In [8]:
from sklearn.linear_model import LogisticRegression

clf_unigrams = LogisticRegression(max_iter=1000) # Instantiate a logistic regression classifier
clf_unigrams.fit(train_features, y_train) # Train the classifier

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [9]:
# Evaluate unigram logistic regression classifier
from sklearn.metrics import classification_report # this provides a bunch of useful evaluation metrics

unigram_predictions = clf_unigrams.predict(test_features)

results = pd.DataFrame(classification_report(y_test, unigram_predictions, output_dict=True))
results

,American,Chinese,French,Italian,Japanese,Korean,Mediterranean,Mexican,Vietnamese,accuracy,macro avg,weighted avg
precision,0.695312,0.557692,0.687500,0.601852,0.787879,0.500000,0.750000,0.659574,0.750000,0.681637,0.665534,0.680155
recall,0.888224,0.460317,0.343750,0.467626,0.641975,0.250000,0.230769,0.563636,0.300000,0.681637,0.460700,0.681637
f1-score,0.780018,0.504348,0.458333,0.526316,0.707483,0.333333,0.352941,0.607843,0.428571,0.681637,0.522132,0.661491
support,501.000000,63.000000,32.000000,139.000000,81.000000,4.000000,52.000000,110.000000,20.000000,0.681637,1002.000000,1002.000000
